**Create Table**

In [ ]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRESQL
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

cursor = conn.cursor()

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """

CREATE TABLE IF NOT EXISTS Official_Holidays (

    Holiday_Date DATE PRIMARY KEY,

    Holiday_Name TEXT NOT NULL,

    Source TEXT,

    Created_At TIMESTAMP DEFAULT CURRENT_TIMESTAMP

);

"""

cursor.execute(create_table_query)

# =========================================
# CREATE INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_official_holidays_name

ON Official_Holidays

(
    Holiday_Name
);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_official_holidays_source

ON Official_Holidays

(
    Source
);

""")

# =========================================
# COMMIT CHANGES
# =========================================

conn.commit()

print(
    "Official_Holidays table created successfully."
)

print(
    "Indexes created successfully."
)

# =========================================
# VERIFY TABLE STRUCTURE
# =========================================

cursor.execute("""

SELECT

    column_name,
    data_type

FROM information_schema.columns

WHERE table_name = 'official_holidays'

ORDER BY ordinal_position;

""")

print("\nOfficial_Holidays Schema:")

for row in cursor.fetchall():

    print(row)

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()

conn.close()

print(
    "\nDatabase connection closed."
)

Official_Holidays table created successfully.
Indexes created successfully.

Official_Holidays Schema:
('holiday_date', 'date')
('holiday_name', 'text')
('source', 'text')
('created_at', 'timestamp without time zone')

Database connection closed.


**Insert Data for First Time**

In [ ]:
import sys
import psycopg2
import pandas as pd

sys.path.append("..")

from Keys.GCalendarAPIKey import GCalendarAPI
from Keys.PostGresKey import POSTGRES_URL

from googleapiclient.discovery import build

# =========================================
# CONFIG
# =========================================

TARGET_YEAR = 2026

CALENDAR_ID = (
    "en.eg.official#holiday@group.v.calendar.google.com"
)

# =========================================
# GOOGLE CALENDAR
# =========================================

service = build(
    "calendar",
    "v3",
    developerKey=GCalendarAPI
)

events = service.events().list(

    calendarId=CALENDAR_ID,

    timeMin=f"{TARGET_YEAR}-01-01T00:00:00Z",

    timeMax=f"{TARGET_YEAR}-12-31T23:59:59Z",

    singleEvents=True,

    orderBy="startTime"

).execute()

rows = []

for event in events.get("items", []):

    holiday_date = event["start"].get("date")

    if not holiday_date:
        continue

    rows.append({

        "holiday_date": holiday_date,

        "holiday_name": event["summary"]

    })

df = pd.DataFrame(rows)

df = df.drop_duplicates(
    subset=["holiday_date"]
)

print(df)

print(
    f"\nTotal Holidays: {len(df)}"
)

# =========================================
# POSTGRES
# =========================================

conn = psycopg2.connect(
    POSTGRES_URL
)

cursor = conn.cursor()

for _, row in df.iterrows():

    cursor.execute(

        """

        INSERT INTO Official_Holidays

        (

            Holiday_Date,
            Holiday_Name,
            Source

        )

        VALUES

        (

            %s,
            %s,
            %s

        )

        ON CONFLICT
        (Holiday_Date)

        DO UPDATE

        SET

            Holiday_Name =
            EXCLUDED.Holiday_Name,

            Source =
            EXCLUDED.Source

        """,

        (

            row["holiday_date"],

            row["holiday_name"],

            "Google Calendar"

        )

    )

conn.commit()

cursor.close()
conn.close()

print(
    "\n2026 Holidays Loaded Successfully"
)